# D3 Flood Analysis Workflow Notebook

This notebook provides section-by-section flood screening and scenario analysis with real API connector stubs and deterministic fallback.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt

OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_LIVE_API = os.getenv('GEOPROMPT_ALLOW_LIVE_API', '0') == '1'


def fetch_json(url: str, fallback: dict) -> dict:
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={'User-Agent': 'geoprompt-section-d-notebook/2.0'})
        with urlopen(req, timeout=10) as response:  # nosec B310
            return json.loads(response.read().decode('utf-8'))
    except (URLError, TimeoutError, ValueError):
        return fallback


import geoprompt as gp
from geoprompt import GeoPromptFrame, frame_to_geojson, write_geojson
from geoprompt.raster import raster_slope_aspect, raster_hillshade
from geoprompt.tools import build_scenario_report, export_scenario_report

## Section A: Pull Flood Data Sources


In [2]:
usgs = fetch_json('https://waterservices.usgs.gov/nwis/iv/?format=json&sites=10109000', {'value': {'timeSeries': []}})
fema = fetch_json('https://hazards.fema.gov/gis/nfhl/rest/services/public/NFHL/MapServer?f=pjson', {'layers': []})
noaa = fetch_json('https://api.weather.gov/points/29.76,-95.37', {'properties': {'forecast': None}})
print('USGS time series:', len(usgs.get('value', {}).get('timeSeries', [])))
print('FEMA layers:', len(fema.get('layers', [])))
print('NOAA forecast pointer exists:', bool(noaa.get('properties', {}).get('forecast')))


USGS time series: 0
FEMA layers: 0
NOAA forecast pointer exists: False


## Section B: Simple Track


In [ ]:
RAW_ASSETS = [
    {'asset_id': 'A1', 'zone': 'X',  'replacement_musd': 4.0, 'elevation_m': 11.0,
     'geometry': {'type': 'Point', 'coordinates': [-95.42, 29.81]}},
    {'asset_id': 'A2', 'zone': 'AE', 'replacement_musd': 7.5, 'elevation_m': 7.0,
     'geometry': {'type': 'Point', 'coordinates': [-95.35, 29.76]}},
    {'asset_id': 'A3', 'zone': 'AE', 'replacement_musd': 3.2, 'elevation_m': 8.0,
     'geometry': {'type': 'Point', 'coordinates': [-95.29, 29.73]}},
    {'asset_id': 'A4', 'zone': 'VE', 'replacement_musd': 9.1, 'elevation_m': 5.9,
     'geometry': {'type': 'Point', 'coordinates': [-95.26, 29.68]}},
]
ZONE_RISK = {'VE': 1.0, 'AE': 0.8, 'X': 0.2}

# Compute risk scores from plain dicts — no pandas Scalar type issues.
enriched_rows = []
for row in RAW_ASSETS:
    zone_risk = ZONE_RISK.get(str(row['zone']), 0.3)
    elev = float(row['elevation_m'])
    risk = round(zone_risk * 0.7 + (10.0 / elev) * 0.3, 4)
    loss = round(float(row['replacement_musd']) * risk * 0.18, 4)
    enriched_rows.append({**row, 'flood_risk_score': risk, 'expected_loss_musd': loss})

# Build GeoPromptFrame — native spatial structure.
assets_frame = GeoPromptFrame(enriched_rows, geometry_column='geometry', crs='EPSG:4326')

# Spatial analysis: nearest neighbors between flood-exposed assets.
neighbors = assets_frame.nearest_neighbors(id_column='asset_id', k=1, distance_method='haversine')
print('--- Asset flood risk summary ---')
print(json.dumps(assets_frame.summary(), indent=2, default=str))
print('\n--- Nearest neighbor pairs ---')
for nb in neighbors:
    print(f"  {nb['origin']} → {nb['neighbor']}  dist={nb['distance']:.5f}°")

# Write GeoJSON map using GeoPrompt.
map_geojson_path = OUTPUT_DIR / 'd3-notebook-flood-exposure-map.geojson'
write_geojson(map_geojson_path, assets_frame)
print(f'\nWrote GeoJSON map: {map_geojson_path}')

# ── Inline scatter map ──────────────────────────────────────────────────────
records = assets_frame.to_records()
lons = [float(r['geometry']['coordinates'][0]) for r in records]
lats = [float(r['geometry']['coordinates'][1]) for r in records]
risks = [float(r['flood_risk_score']) for r in records]
losses = [float(r['expected_loss_musd']) for r in records]
asset_ids = [str(r['asset_id']) for r in records]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(lons, lats, c=risks, cmap='Blues', s=200, edgecolors='#1d4ed8', zorder=5)
for lon, lat, aid, risk in zip(lons, lats, asset_ids, risks):
    axes[0].annotate(f'{aid}\n({risk:.2f})', (lon, lat), textcoords='offset points',
                     xytext=(5, 5), fontsize=9)
plt.colorbar(sc, ax=axes[0], label='Flood Risk Score')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('D3 Flood Exposure Map')
axes[0].grid(True, alpha=0.3)

sorted_records = sorted(records, key=lambda r: -float(r['flood_risk_score']))
bar_labels = [str(r['asset_id']) for r in sorted_records]
bar_risks = [float(r['flood_risk_score']) for r in sorted_records]
bar_losses = [float(r['expected_loss_musd']) for r in sorted_records]
bar_colors = ['#c0392b' if v > 0.8 else '#e67e22' if v > 0.5 else '#27ae60' for v in bar_risks]

x = range(len(bar_labels))
width = 0.38
axes[1].bar([i - width/2 for i in x], bar_risks, width=width, color=bar_colors, label='Flood Risk Score')
ax2 = axes[1].twinx()
ax2.bar([i + width/2 for i in x], bar_losses, width=width, color='#94a3b8', alpha=0.7, label='Expected Loss $M')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(bar_labels)
axes[1].set_ylabel('Flood Risk Score')
ax2.set_ylabel('Expected Loss ($M USD)')
axes[1].set_title('Flood Risk and Expected Loss by Asset')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')
axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Write summary JSON.
simple_payload = {
    'track': 'simple',
    'assets': records,
    'flood_map_geojson': str(map_geojson_path),
}
(OUTPUT_DIR / 'd3-notebook-simple.json').write_text(json.dumps(simple_payload, indent=2, default=str), encoding='utf-8')
print('Wrote d3-notebook-simple.json')

GeoPrompt map feature count: 4


,asset_id,zone,replacement_musd,elevation_m,lon,lat,flood_risk_score,expected_loss_musd
0,A1,X,4.0,11.0,-95.42,29.81,0.4127,0.2971
1,A2,AE,7.5,7.0,-95.35,29.76,0.9886,1.3346
2,A3,AE,3.2,8.0,-95.29,29.73,0.9350,0.5386
3,A4,VE,9.1,5.9,-95.26,29.68,1.2085,1.9795


Wrote D:\Github\geoprompt\outputs\d3-notebook-simple.json
Wrote D:\Github\geoprompt\outputs\d3-notebook-flood-exposure-map.geojson
Wrote D:\Github\geoprompt\outputs\d3-notebook-flood-exposure-map.html


## Section C: Complex Track


In [ ]:
raster = {
    'data': [[1.0, 2.0, 2.0], [2.0, 3.0, 4.0], [3.0, 4.0, 6.0]],
    'transform': (0.0, 3.0, 1.0, 1.0),
}
slope = raster_slope_aspect(raster)
hillshade = raster_hillshade(raster)

baseline = {'expected_annual_loss': 18.0, 'impacted_assets': 3, 'response_time_hours': 8.0}
mitigation = {'expected_annual_loss': 10.5, 'impacted_assets': 2, 'response_time_hours': 6.0}
report = build_scenario_report(baseline, mitigation, higher_is_better=[])
report_path = export_scenario_report(report, OUTPUT_DIR / 'd3-notebook-scenario-report.json')
print('Scenario report:', report_path)

# ── Inline scenario comparison chart ─────────────────────────────────────────
metrics = list(baseline.keys())
baseline_vals = [float(baseline[m]) for m in metrics]
mitigation_vals = [float(mitigation[m]) for m in metrics]
delta_pct = [round((m - b) / b * 100, 1) for b, m in zip(baseline_vals, mitigation_vals)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = range(len(metrics))
width = 0.38
axes[0].bar([i - width/2 for i in x], baseline_vals, width=width, label='Baseline', color='#94a3b8')
axes[0].bar([i + width/2 for i in x], mitigation_vals, width=width, label='Mitigation', color='#2563eb')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(metrics, rotation=15)
axes[0].set_title('Metric Comparison: Baseline vs Mitigation')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

bar_colors = ['#27ae60' if d < 0 else '#c0392b' for d in delta_pct]
axes[1].barh(metrics, delta_pct, color=bar_colors)
axes[1].axvline(0, color='#555', linewidth=1)
axes[1].set_xlabel('% Change')
axes[1].set_title('Delta % (negative = improvement)')
axes[1].grid(True, axis='x', alpha=0.3)

plt.suptitle('D3 Flood Mitigation — Scenario Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

# Inline raster terrain visualization.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
hs_grid = hillshade['grid']
sl_grid = slope['slope']
im0 = axes[0].imshow(hs_grid, cmap='gray', origin='upper')
plt.colorbar(im0, ax=axes[0], label='Hillshade')
axes[0].set_title('Terrain Hillshade')
im1 = axes[1].imshow(sl_grid, cmap='RdYlGn_r', origin='upper')
plt.colorbar(im1, ax=axes[1], label='Slope (deg)')
axes[1].set_title('Terrain Slope')
plt.suptitle('Raster Terrain Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

complex_payload = {
    'track': 'complex',
    'slope_center': slope['slope'][1][1],
    'hillshade_center': hillshade['grid'][1][1],
    'scenario_report_path': str(report_path),
}
(OUTPUT_DIR / 'd3-notebook-complex.json').write_text(json.dumps(complex_payload, indent=2, default=str), encoding='utf-8')
print('Wrote d3-notebook-complex.json')

Wrote D:\Github\geoprompt\outputs\d3-notebook-complex.json
